In [11]:
# Setup
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML

sys.path.insert(0, str(Path.cwd().parent))

from analysis.bluechip_quality import (
    find_bluechip_quality,
    find_bluechip_dips,
    _quality_pe_cap,
    _entry_signal,
)

from notebooks.nb_helpers import (
    load_report_cache,
    get_batch_report_insights,
    load_or_fetch_news,
    get_news_sentiment,
    data_freshness_banner,
    TONE_EMOJI,
    print_divergence_warnings,
    print_report_block,
    print_news_block,
)

report_cache = load_report_cache()

DATA_DIR = Path(r'c:/Users/chris/Desktop/South_African_Stocks/data') / 'snapshots'

print(f"📅 Report Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"📁 Data Directory: {DATA_DIR}")


⚠️  No report cache files found in data/reports/.
📅 Report Date: 2026-05-15 17:37
📁 Data Directory: c:\Users\chris\Desktop\South_African_Stocks\data\snapshots


## 📊 Load Latest Snapshot

In [12]:
# Find and load the latest snapshot
snapshots = sorted(DATA_DIR.glob('*/snapshot.parquet'))

if not snapshots:
    print("❌ No snapshots found! Run create_snapshot.py first.")
    df = pd.DataFrame()
    snapshot_date = None
else:
    latest = snapshots[-1]
    snapshot_date = latest.parent.name
    df = pd.read_parquet(latest)

    if 'volume_1d' in df.columns:
        df['liquidity_value_1d'] = df['price'] * df['volume_1d']

    print(f"✅ Loaded snapshot: {snapshot_date}")
    print(f"📈 Total stocks: {len(df)}")
    print(f"📊 Total columns: {len(df.columns)}")
    print(f"\nAvailable snapshots:")
    for s in snapshots[-8:]:
        marker = "👉" if s == latest else "  "
        print(f"  {marker} {s.parent.name}")


✅ Loaded snapshot: 2026-05-15
📈 Total stocks: 245
📊 Total columns: 105

Available snapshots:
  👉 2026-05-15


## 🏆 Blue-Chip Quality Screen

Strong **fundamentals** (ROE, margin, EPS growth) regardless of market cap.
Catches names like **SBK, NPN, SHP, MTN** that hidden_gems
misses because they are never "cold" and their P/E often exceeds the v2.1 cap.

Safety gates (v2.1 compatible):
- **Dynamic P/E cap** — high-ROE businesses earn higher P/E (function `_quality_pe_cap`)
- **Liquidity floor** — R10M+ daily turnover
- **Falling-knife guard** — 3M perf > -30%
- **Value-trap detection** — reused from hidden_gems


In [13]:
bluechips = find_bluechip_quality(df, top_n=25)

if len(bluechips) == 0:
    print("⚠️  No blue-chip quality stocks passed the filters on this snapshot.")
else:
    elite   = (bluechips['tier'] == '🏆 Elite').sum()
    premium = (bluechips['tier'] == '⭐ Premium').sum()
    quality = (bluechips['tier'] == '✅ Quality').sum()
    watch   = (bluechips['tier'] == '📊 Watchlist').sum()

    print(f"\n{'='*70}")
    print(f"🏆 BLUE-CHIP QUALITY SUMMARY")
    print(f"{'='*70}")
    print(f"\n  Total found        : {len(bluechips)}")
    print(f"  🏆 Elite     (≥70) : {elite}")
    print(f"  ⭐ Premium (55-69) : {premium}")
    print(f"  ✅ Quality (40-54) : {quality}")
    print(f"  📊 Watchlist (<40) : {watch}")



🏆 BLUE-CHIP QUALITY SUMMARY

  Total found        : 25
  🏆 Elite     (≥70) : 6
  ⭐ Premium (55-69) : 5
  ✅ Quality (40-54) : 14
  📊 Watchlist (<40) : 0


In [14]:
if len(bluechips) > 0:
    display_cols = [
        'rank', 'symbol', 'tier', 'price', 'bluechip_score',
        'pe_ratio', 'pe_cap', 'rsi_14',
        'perf_1m', 'perf_3m', 'perf_ytd',
        'eps_growth_ttm', 'roe_ttm', 'net_margin_ttm',
        'entry_signal', 'warnings'
    ]
    available = [c for c in display_cols if c in bluechips.columns]
    display_df = bluechips[available].copy()
    display_df = display_df.rename(columns={
        'rank':           'Rank',
        'symbol':         'Symbol',
        'tier':           'Tier',
        'price':          'Price',
        'bluechip_score': 'Score',
        'pe_ratio':       'P/E',
        'pe_cap':         'PE Cap',
        'rsi_14':         'RSI',
        'perf_1m':        '1M%',
        'perf_3m':        '3M%',
        'perf_ytd':       'YTD%',
        'eps_growth_ttm': 'EPS Gr',
        'roe_ttm':        'ROE',
        'net_margin_ttm': 'Margin',
        'entry_signal':   'Entry Signal',
        'warnings':       'Warnings',
    })

    numeric_cols = ['Score', 'P/E', 'PE Cap', 'RSI', '1M%', '3M%', 'YTD%',
                    'EPS Gr', 'ROE', 'Margin']
    for c in numeric_cols:
        if c in display_df.columns:
            display_df[c] = display_df[c].round(1)

    print("\n🔝 TOP BLUE-CHIP QUALITY STOCKS:\n")
    display(display_df.style.background_gradient(subset=['Score'], cmap='Greens'))
else:
    print("No blue-chip data to display.")



🔝 TOP BLUE-CHIP QUALITY STOCKS:



,Rank,Symbol,Tier,Price,Score,P/E,PE Cap,RSI,1M%,3M%,YTD%,EPS Gr,ROE,Margin,Entry Signal,Warnings
9,1,GFI,🏆 Elite,68648.000000,96.600000,9.800000,25.000000,38.500000,-17.000000,-19.400000,-7.100000,177.800000,53.000000,40.500000,🟢 BUY ZONE,No analyst coverage
43,2,PAN,🏆 Elite,3084.000000,91.800000,15.100000,25.000000,40.400000,-13.700000,1.100000,13.000000,157.500000,43.500000,29.100000,🟢 BUY ZONE,No analyst coverage
57,3,DRD,🏆 Elite,4527.000000,80.400000,12.300000,22.300000,38.300000,-14.300000,-16.300000,-14.700000,86.500000,34.700000,35.000000,🟢 BUY ZONE,No analyst coverage
5,4,PRX,🏆 Elite,76368.000000,74.900000,7.100000,21.000000,41.400000,-5.800000,-6.900000,-27.300000,91.400000,27.300000,198.600000,🟢 BUY ZONE,Zero payout | No analyst coverage
7,5,ANG,🏆 Elite,158759.000000,74.800000,13.700000,20.000000,44.200000,-11.000000,-6.700000,10.600000,nan,45.400000,31.100000,🟢 BUY ZONE,No analyst coverage
26,6,NPH,🏆 Elite,33915.000000,70.200000,14.800000,22.200000,45.900000,-4.800000,-9.300000,-0.800000,496.600000,26.200000,22.200000,🔵 ACCUMULATE,No analyst coverage
8,7,NPN,⭐ Premium,86489.000000,69.600000,6.900000,20.600000,43.500000,-6.000000,-4.500000,-23.700000,85.400000,26.700000,73.000000,🟢 BUY ZONE,Zero payout | No analyst coverage
2,8,BTI,⭐ Premium,108984.000000,66.800000,13.900000,20.100000,73.000000,16.000000,14.900000,13.000000,144.600000,15.700000,30.000000,🔴 WAIT — OVERBOUGHT,RSI 73 overbought | No analyst coverage
21,9,HAR,⭐ Premium,27140.000000,62.800000,10.500000,20.400000,45.700000,-2.500000,-17.300000,-19.700000,54.300000,33.300000,18.300000,🔵 ACCUMULATE,No analyst coverage
11,10,FSR,⭐ Premium,8784.000000,57.000000,11.200000,16.300000,46.400000,-2.300000,-7.700000,-2.500000,10.700000,21.300000,29.600000,🔵 ACCUMULATE,No analyst coverage


## 💎 Blue-Chip Dip Opportunities

Quality names currently **pulling back** (negative 1-month) — the BEST entry points.
Like catching SBK at R680 or NPN at R183.


In [15]:
dips = find_bluechip_dips(df, top_n=15)

def _tier_for_score(s):
    if s >= 70: return '🏆 Elite'
    if s >= 55: return '⭐ Premium'
    if s >= 40: return '✅ Quality'
    return '📊 Watchlist'

if len(dips) > 0 and 'tier' not in dips.columns:
    dips['tier'] = dips['bluechip_score'].apply(_tier_for_score)

if len(dips) == 0:
    print("\n⚠️  No blue-chip dips on this snapshot.")
    print("    The market is hot — quality names are running.")
    print("    Strategies:")
    print("      1. Wait for a market pullback")
    print("      2. Use staggered entries on Premium/Elite tier names")
    print("      3. Take profits in any overbought blue-chips you hold")
else:
    print(f"\n{'='*70}")
    print(f"💎 BLUE-CHIP DIP OPPORTUNITIES — {len(dips)} quality stocks on pullback")
    print(f"{'='*70}\n")

    for _, row in dips.iterrows():
        # Technical confluence — same idea as hidden_gems cold opportunities
        tech_signals = []
        stoch_k = row.get('stochastic_k_1d', np.nan)
        cci     = row.get('cci_20_1d', np.nan)
        rvol    = row.get('relative_volume_1d', np.nan)

        if not pd.isna(stoch_k) and stoch_k < 20:
            tech_signals.append(f"Stoch K={stoch_k:.0f}")
        if not pd.isna(cci) and cci < -100:
            tech_signals.append(f"CCI={cci:.0f}")
        if not pd.isna(rvol) and rvol > 2.0:
            tech_signals.append(f"🔊 RVOL={rvol:.1f}x")

        confluence = '⚡ OVERSOLD CONFIRMED' if len(tech_signals) >= 2 else ''

        pe = row.get('pe_ratio', np.nan)
        pe_str = f"{pe:.1f}x" if not pd.isna(pe) else "—"
        cap_str = f"{row['pe_cap']:.0f}x"

        print(f"🎯 {row['symbol']}  {row['tier']}  {confluence}")
        print(f"   Price: R{row['price']:.2f} | Score: {row['bluechip_score']:.1f}")
        print(f"   1M: {row['perf_1m']:+.1f}% | 3M: {row.get('perf_3m', 0):+.1f}% | "
              f"YTD: {row.get('perf_ytd', 0):+.1f}%")
        print(f"   ROE: {row['roe_ttm']:.1f}% | Margin: {row['net_margin_ttm']:.1f}% | "
              f"EPS Gr: {row.get('eps_growth_ttm', 0):.1f}%")
        print(f"   P/E: {pe_str} (cap {cap_str}) | Entry: {row['entry_signal']}")
        if tech_signals:
            print(f"   📊 Tech: {' | '.join(tech_signals)}")
        if row.get('warnings'):
            print(f"   ⚠️  {row['warnings']}")
        print()



💎 BLUE-CHIP DIP OPPORTUNITIES — 15 quality stocks on pullback

🎯 GFI  🏆 Elite  
   Price: R68648.00 | Score: 96.6
   1M: -17.0% | 3M: -19.4% | YTD: -7.1%
   ROE: 53.0% | Margin: 40.5% | EPS Gr: 177.8%
   P/E: 9.8x (cap 25x) | Entry: 🟢 BUY ZONE
   📊 Tech: CCI=-125
   ⚠️  No analyst coverage

🎯 PAN  🏆 Elite  
   Price: R3084.00 | Score: 91.8
   1M: -13.7% | 3M: +1.1% | YTD: +13.0%
   ROE: 43.5% | Margin: 29.1% | EPS Gr: 157.5%
   P/E: 15.1x (cap 25x) | Entry: 🟢 BUY ZONE
   📊 Tech: CCI=-109
   ⚠️  No analyst coverage

🎯 DRD  🏆 Elite  
   Price: R4527.00 | Score: 80.4
   1M: -14.3% | 3M: -16.3% | YTD: -14.7%
   ROE: 34.7% | Margin: 35.0% | EPS Gr: 86.5%
   P/E: 12.3x (cap 22x) | Entry: 🟢 BUY ZONE
   ⚠️  No analyst coverage

🎯 PRX  🏆 Elite  
   Price: R76368.00 | Score: 74.9
   1M: -5.8% | 3M: -6.9% | YTD: -27.3%
   ROE: 27.3% | Margin: 198.6% | EPS Gr: 91.4%
   P/E: 7.1x (cap 21x) | Entry: 🟢 BUY ZONE
   📊 Tech: CCI=-134
   ⚠️  Zero payout | No analyst coverage

🎯 ANG  🏆 Elite  
   Price: 

## 🌍 Sector Quality Heatmap

In [16]:
if len(bluechips) > 0 and 'sector' in bluechips.columns:
    def _sector_stats(group):
        top = group.sort_values('bluechip_score', ascending=False).iloc[0]
        return pd.Series({
            'n':          len(group),
            'avg_score':  group['bluechip_score'].mean(),
            'avg_roe':    group['roe_ttm'].mean(),
            'avg_margin': group['net_margin_ttm'].mean(),
            'best':       top['symbol'],
            'best_score': top['bluechip_score'],
        })

    sector_summary = (
        bluechips.groupby('sector', dropna=False)
                 .apply(_sector_stats, include_groups=False)
                 .sort_values('avg_score', ascending=False)
    )

    print(f"\n{'='*90}")
    print(f"🌍 SECTOR QUALITY HEATMAP — top sectors by average blue-chip score")
    print(f"{'='*90}")
    print(f"  {'Sector':32} | {'#':>3} | {'Avg Score':>9} | {'Avg ROE':>8} | "
          f"{'Avg Margin':>10} | {'Best Stock':12} | {'Score':>5}")
    print(f"  {'-'*90}")
    for sector, r in sector_summary.iterrows():
        sec_name = (sector or 'Unknown')[:32]
        print(f"  {sec_name:32} | {int(r['n']):>3} | {r['avg_score']:>9.1f} | "
              f"{r['avg_roe']:>7.1f}% | {r['avg_margin']:>9.1f}% | "
              f"{r['best']:12} | {r['best_score']:>5.1f}")
else:
    print("No sector data available.")



🌍 SECTOR QUALITY HEATMAP — top sectors by average blue-chip score
  Sector                           |   # | Avg Score |  Avg ROE | Avg Margin | Best Stock   | Score
  ------------------------------------------------------------------------------------------
  Non-energy minerals              |   6 |      79.4 |    39.4% |      29.4% | GFI          |  96.6
  Technology services              |   3 |      65.9 |    32.0% |      92.4% | PRX          |  74.9
  Consumer services                |   1 |      56.0 |    53.0% |      15.1% | SUI          |  56.0
  Consumer non-durables            |   3 |      54.1 |    24.3% |      16.3% | BTI          |  66.8
  Process industries               |   1 |      52.3 |    19.5% |       5.5% | RBO          |  52.3
  Finance                          |   5 |      52.1 |    19.3% |      27.2% | FSR          |  57.0
  Retail trade                     |   4 |      47.0 |    33.3% |       6.7% | WBC          |  51.0
  Communications                   |   1

## 🔁 Quality vs Value Comparison

Which stocks appear in **both** the hidden_gems and blue-chip screens? Which are
exclusive to each? This shows the real **coverage gap** between the two models.


In [17]:
from analysis.hidden_gems import find_hidden_gems

gems_30      = find_hidden_gems(df, top_n=30)
bluechips_30 = find_bluechip_quality(df, top_n=30)

gem_set      = set(gems_30['symbol'])      if len(gems_30)      > 0 else set()
bluechip_set = set(bluechips_30['symbol']) if len(bluechips_30) > 0 else set()

both          = gem_set & bluechip_set
gems_only     = gem_set - both
bluechip_only = bluechip_set - both

print(f"\n{'='*70}")
print(f"🔁 QUALITY vs VALUE — coverage map (top 30 in each model)")
print(f"{'='*70}")
print(f"\n  💎 Hidden gems     : {len(gem_set):2}")
print(f"  🏆 Blue-chips      : {len(bluechip_set):2}")
print(f"  🟢 Appear in BOTH  : {len(both):2}  — best of both worlds")
print(f"  💎 Gems only       : {len(gems_only):2}  — small/mid-cap contrarian plays")
print(f"  🏆 Blue-chip only  : {len(bluechip_only):2}  — institutional quality not seen by gems")

def _show_set(label, syms, source_df, score_col):
    if not syms:
        print(f"\n  {label}: (none)")
        return
    print(f"\n  {label}:")
    sub = source_df[source_df['symbol'].isin(syms)].sort_values(score_col, ascending=False)
    for _, r in sub.iterrows():
        score = r[score_col]
        sec   = r.get('sector', '') or ''
        p1m   = r.get('perf_1m', 0) or 0
        print(f"    • {r['symbol']:12} score={score:5.1f}  1M={p1m:+5.1f}%  {sec}")

_show_set("🟢 BOTH (highest conviction)", both,          bluechips_30, 'bluechip_score')
_show_set("💎 Gems only",                  gems_only,     gems_30,      'hidden_gem_score')
_show_set("🏆 Blue-chip only",             bluechip_only, bluechips_30, 'bluechip_score')



🔁 QUALITY vs VALUE — coverage map (top 30 in each model)

  💎 Hidden gems     : 30
  🏆 Blue-chips      : 30
  🟢 Appear in BOTH  : 16  — best of both worlds
  💎 Gems only       : 14  — small/mid-cap contrarian plays
  🏆 Blue-chip only  : 14  — institutional quality not seen by gems

  🟢 BOTH (highest conviction):
    • GFI          score= 96.6  1M=-17.0%  Non-energy minerals
    • PAN          score= 91.8  1M=-13.7%  Non-energy minerals
    • DRD          score= 80.4  1M=-14.3%  Non-energy minerals
    • PRX          score= 74.9  1M= -5.8%  Technology services
    • ANG          score= 74.8  1M=-11.0%  Non-energy minerals
    • NPH          score= 70.2  1M= -4.8%  Non-energy minerals
    • NPN          score= 69.6  1M= -6.0%  Technology services
    • HAR          score= 62.8  1M= -2.5%  Non-energy minerals
    • SUI          score= 56.0  1M= +1.6%  Consumer services
    • FTB          score= 54.5  1M= -1.6%  Finance
    • AFH          score= 54.2  1M=+11.7%  Finance
    • IOC         

## 📋 My Portfolio — Blue-Chip Lens

Same watchlist as hidden_gems, but viewed through quality metrics: ROE, margin,
EPS growth, dynamic P/E cap, RSI entry signal, and the blue-chip tier (if it
qualifies on the latest screen).


In [18]:
WATCHLIST = [
    '# removed', '# removed', '# removed', '# removed', 'NPN',
    '# removed', '# removed', '# removed', '# removed', '# removed',
    '# removed', '# removed', '# removed', '# removed', '# removed',
    '# removed', '# removed', '# removed', '# removed', '# removed',
    '# removed', '# removed', '# removed', '# removed', '# removed', '# removed',
]

# Full blue-chip universe (no top_n cap) so we can attach tier to every match
all_bluechips = find_bluechip_quality(df, top_n=10_000)
tier_lookup   = dict(zip(all_bluechips['symbol'], all_bluechips['tier'])) if len(all_bluechips) else {}

print(f"\n{'='*120}")
print(f"📋 MY PORTFOLIO — BLUE-CHIP LENS  ({len(WATCHLIST)} positions)")
print(f"{'='*120}")
print(f"{'Symbol':12} | {'Price':>9} | {'P/E':>7} | {'PE Cap':>7} | "
      f"{'ROE':>6} | {'Margin':>7} | {'EPS Gr':>7} | {'RSI':>5} | "
      f"{'Entry Signal':22} | Tier")
print(f"{'-'*120}")

for symbol in WATCHLIST:
    stock = df[df['symbol'] == symbol]
    if len(stock) == 0:
        print(f"{symbol:12} | not found in snapshot")
        continue

    row = stock.iloc[0]
    roe   = row.get('roe_ttm', np.nan)
    eps_g = row.get('eps_growth_ttm', np.nan)
    pe    = row.get('pe_ratio', np.nan)
    nm    = row.get('net_margin_ttm', np.nan)
    rsi   = row.get('rsi_14', np.nan)
    cap   = _quality_pe_cap(roe, eps_g)
    sig   = _entry_signal(row)
    tier  = tier_lookup.get(symbol, '—')

    pe_str    = f"{pe:5.1f}x"  if not pd.isna(pe)    else "    —"
    cap_str   = f"{cap:5.1f}x"
    roe_str   = f"{roe:5.1f}%" if not pd.isna(roe)   else "    —"
    nm_str    = f"{nm:6.1f}%"  if not pd.isna(nm)    else "     —"
    epsg_str  = f"{eps_g:+6.1f}" if not pd.isna(eps_g) else "     —"
    rsi_str   = f"{rsi:5.1f}"  if not pd.isna(rsi)   else "    —"

    print(f"{symbol:12} | R{row['price']:8.2f} | {pe_str} | {cap_str} | "
          f"{roe_str} | {nm_str} | {epsg_str} | {rsi_str} | {sig:22} | {tier}")

print()
print("  Tier key: 🏆 Elite ≥70  ·  ⭐ Premium 55-69  ·  ✅ Quality 40-54  ·  📊 Watchlist <40")
print("  P/E is allowed up to PE Cap (cap scales with ROE & EPS growth).")



📋 MY PORTFOLIO — BLUE-CHIP LENS  (26 positions)
Symbol       |     Price |     P/E |  PE Cap |    ROE |  Margin |  EPS Gr |   RSI | Entry Signal           | Tier
------------------------------------------------------------------------------------------------------------------------
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
NPN          | R86489.00 |   6.9x |  20.6x |  26.7% |   73.0% |  +85.4 |  43.5 | 🟢 BUY ZONE             | ⭐ Premium
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# removed    | not found in snapshot
# remove

## 📈 RSI Signals — Market-Wide

In [19]:
# RSI Signals — Overbought / Oversold across full market
RSI_COL = 'rsi_14'

rsi_df = df[df[RSI_COL].notna()][['symbol', 'sector', 'price', 'perf_1m', 'perf_1w', RSI_COL]].copy()
rsi_df = rsi_df.rename(columns={RSI_COL: 'rsi'})

oversold   = rsi_df[rsi_df['rsi'] <= 30].sort_values('rsi')
overbought = rsi_df[rsi_df['rsi'] >= 70].sort_values('rsi', ascending=False)

# --- OVERSOLD (potential buy / entry zone) ---
print(f"\n{'='*70}")
print(f"🟢 OVERSOLD — RSI ≤ 30  (potential entry zone)")
print(f"{'='*70}")
if len(oversold):
    print(f"{'Symbol':12} | {'RSI':>5} | {'Price':>9} | {'1W%':>7} | {'1M%':>7} | Sector")
    print(f"{'-'*70}")
    for _, r in oversold.iterrows():
        watch = "⭐" if r['symbol'] in WATCHLIST else "  "
        print(f"{watch}{r['symbol']:10} | {r['rsi']:5.1f} | R{r['price']:8.2f} | "
              f"{r['perf_1w']:+6.1f}% | {r['perf_1m']:+6.1f}% | {r.get('sector','')}")
else:
    print("  None currently — market not in oversold territory")

# --- OVERBOUGHT (consider taking profit) ---
print(f"\n{'='*70}")
print(f"🔴 OVERBOUGHT — RSI ≥ 70  (consider taking profit / avoid new entries)")
print(f"{'='*70}")
if len(overbought):
    print(f"{'Symbol':12} | {'RSI':>5} | {'Price':>9} | {'1W%':>7} | {'1M%':>7} | Sector")
    print(f"{'-'*70}")
    for _, r in overbought.iterrows():
        watch = "⭐" if r['symbol'] in WATCHLIST else "  "
        print(f"{watch}{r['symbol']:10} | {r['rsi']:5.1f} | R{r['price']:8.2f} | "
              f"{r['perf_1w']:+6.1f}% | {r['perf_1m']:+6.1f}% | {r.get('sector','')}")
else:
    print("  None currently")

# --- PORTFOLIO RSI SUMMARY ---
portfolio_rsi = rsi_df[rsi_df['symbol'].isin(WATCHLIST)].sort_values('rsi', ascending=False)
print(f"\n{'='*70}")
print(f"📊 PORTFOLIO RSI DISTRIBUTION")
print(f"{'='*70}")
ob  = portfolio_rsi[portfolio_rsi['rsi'] >= 70]
hi  = portfolio_rsi[(portfolio_rsi['rsi'] >= 60) & (portfolio_rsi['rsi'] < 70)]
ok  = portfolio_rsi[(portfolio_rsi['rsi'] >= 40) & (portfolio_rsi['rsi'] < 60)]
lo  = portfolio_rsi[(portfolio_rsi['rsi'] >= 30) & (portfolio_rsi['rsi'] < 40)]
os_ = portfolio_rsi[portfolio_rsi['rsi'] < 30]
print(f"  🔴 Overbought  (≥70): {len(ob):2}  — {', '.join(ob['symbol'].tolist()) or 'none'}")
print(f"  🟠 High        (60-69): {len(hi):2}  — {', '.join(hi['symbol'].tolist()) or 'none'}")
print(f"  ⚪ Neutral     (40-59): {len(ok):2}  — {', '.join(ok['symbol'].tolist()) or 'none'}")
print(f"  🔵 Low         (31-40): {len(lo):2}  — {', '.join(lo['symbol'].tolist()) or 'none'}")
print(f"  🟢 Oversold    (≤30):   {len(os_):2}  — {', '.join(os_['symbol'].tolist()) or 'none'}")



🟢 OVERSOLD — RSI ≤ 30  (potential entry zone)
Symbol       |   RSI |     Price |     1W% |     1M% | Sector
----------------------------------------------------------------------
  APF        |  13.0 | R   50.00 |  -12.3% |  -19.4% | Finance
  CLS        |  25.5 | R24671.00 |   -6.5% |  -14.9% | Retail trade
  AVI        |  27.6 | R 9442.00 |   -4.6% |   -5.6% | Consumer non-durables
  OPA        |  28.0 | R 1696.00 |   -9.1% |  -21.0% | Finance
  TEX        |  28.2 | R  261.00 |   -6.8% |  -13.3% | Finance
  TFG        |  28.5 | R 5892.00 |  -16.9% |  -20.1% | Retail trade
  CPR        |  28.7 | R   46.00 |  -24.6% |  -29.2% | Non-energy minerals
  MTH        |  29.1 | R10092.00 |   -6.8% |  -12.7% | Distribution services
  WBC        |  29.2 | R 3507.00 |   -6.2% |  -13.1% | Retail trade

🔴 OVERBOUGHT — RSI ≥ 70  (consider taking profit / avoid new entries)
Symbol       |   RSI |     Price |     1W% |     1M% | Sector
-----------------------------------------------------------------

## 📊 Weekly Summary

### Tier playbook
| Tier | Score | What it means | Position sizing |
|------|------:|---------------|-----------------|
| 🏆 Elite      | ≥70   | Best-in-class fundamentals             | 10–15% — accumulate on dips |
| ⭐ Premium    | 55–69 | Strong quality, durable margins        | 8–12% |
| ✅ Quality    | 40–54 | Solid but not exceptional              | 5–8% |
| 📊 Watchlist  | <40   | On the radar — wait for confirmation   | 2–4% probe |

### When to use this notebook vs hidden_gems
- **Blue-chip** for institutional-quality names that compound (SBK, NPN, MTN)
- **Hidden gems** for small/mid-cap contrarian setups (TIP @ R13, # removed @ R9)
- Names appearing in **both** screens are the highest-conviction ideas


In [20]:
print(f"\n{'='*70}")
print(f"📊 BLUE-CHIP WEEKLY SUMMARY — {datetime.now().strftime('%Y-%m-%d')}")
print(f"{'='*70}")
print(f"\n  Snapshot date           : {snapshot_date}")
print(f"  Stocks analyzed         : {len(df)}")
print(f"  Blue-chips found        : {len(bluechips)}")
print(f"  Dip opportunities       : {len(dips)}")

if len(bluechips) > 0:
    by_tier = bluechips['tier'].value_counts()
    print(f"\n  Tier distribution:")
    for tier in ['🏆 Elite', '⭐ Premium', '✅ Quality', '📊 Watchlist']:
        print(f"    {tier:14}: {int(by_tier.get(tier, 0))}")

print(f"\n{'='*70}")
print("\n🔔 Next steps:")
print("  1. Prioritize 🏆 Elite + ⭐ Premium names with 🟢 BUY ZONE entry signals")
print("  2. Stage entries on Blue-Chip Dips — quality on temporary pullback")
print("  3. Cross-check 'BOTH' list (gems ∩ blue-chip) for highest conviction")
print("  4. Re-run after each new TradingView snapshot")



📊 BLUE-CHIP WEEKLY SUMMARY — 2026-05-15

  Snapshot date           : 2026-05-15
  Stocks analyzed         : 245
  Blue-chips found        : 25
  Dip opportunities       : 15

  Tier distribution:
    🏆 Elite       : 6
    ⭐ Premium     : 5
    ✅ Quality     : 14
    📊 Watchlist   : 0


🔔 Next steps:
  1. Prioritize 🏆 Elite + ⭐ Premium names with 🟢 BUY ZONE entry signals
  2. Stage entries on Blue-Chip Dips — quality on temporary pullback
  3. Cross-check 'BOTH' list (gems ∩ blue-chip) for highest conviction
  4. Re-run after each new TradingView snapshot
